# Machine Learning with PySpark  
## E-commerce Dataset Analysis  
### Objective:  
- Build unsupervised and supervised ML models similar to sklearn  
- Predict sales by brand based on cart additions  
- Predict probability of adding to cart based on week_day + hour

In [4]:
# Import libraries
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import countDistinct
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, RegressionEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Create Spark Session
spark = SparkSession.builder \
    .appName("Analisis Ecommerce Octubre") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()
print('Spark Session created successfully!')
spark

Spark Session created successfully!


## 1. Load and Explore Data

In [5]:
# Load datasets
df_oct = spark.read.csv('../datasets/2019-Oct.csv', header=True, inferSchema=True)
df_nov = spark.read.csv('../datasets/2019-Nov.csv', header=True, inferSchema=True)

# Combine datasets
df = df_oct.union(df_nov)

print('Dataset shape:', df.count(), 'rows')
df.printSchema()

Dataset shape: 109950743 rows
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [6]:
# Display sample data
df.show(5)

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-09-30 17:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-09-30 17:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-09-30 17:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-09-30 17:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-09-30 17:00:04|      view|   1004237|2053013555631882655|electr

In [7]:
# Basic statistics
df.describe().show()
print('Column names:', df.columns)
print('Data types:')
df.printSchema()

+-------+----------+--------------------+--------------------+-------------------+--------+------------------+--------------------+--------------------+
|summary|event_type|          product_id|         category_id|      category_code|   brand|             price|             user_id|        user_session|
+-------+----------+--------------------+--------------------+-------------------+--------+------------------+--------------------+--------------------+
|  count| 109950743|           109950743|           109950743|           74536963|94619500|         109950743|           109950743|           109950731|
|   mean|      NULL|1.1755770809475813E7|2.057707154554958...|               NULL|     NaN|291.63480233260026|5.3666978183009773E8|                NULL|
| stddev|      NULL|1.5435644453889327E7|1.949326326906418E16|               NULL|     NaN| 356.6799709149105|2.1451730237962335E7|                NULL|
|    min|      cart|             1000365| 2053013552226107603|    accessories.bag|

In [8]:
# Check for missing values
from pyspark.sql.functions import col, count, when

missing_values = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
missing_values.show()

+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|   brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|         0|         0|         0|          0|     35413780|15331243|    0|      0|          12|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+



## 2. Feature Engineering

In [9]:
# Assuming columns: event_time, product_id, category_id, brand, price, user_id, event_type
# event_type: 'view', 'cart', 'purchase'

# Convert event_time to timestamp and extract temporal features
df_features = df.withColumn('event_time', to_timestamp('event_time', 'yyyy-MM-dd HH:mm:ss')) \
    .withColumn('hour', hour('event_time')) \
    .withColumn('day_of_week', dayofweek('event_time')) \
    .withColumn('day_of_week_name', date_format('event_time', 'EEEE')) \
    .withColumn('week_hour', concat(col('day_of_week'), lit('_'), col('hour')))

# Create binary target for cart events
df_features = df_features.withColumn('added_to_cart', when(col('event_type') == 'cart', 1).otherwise(0))

# Create binary target for purchase events
df_features = df_features.withColumn('purchased', when(col('event_type') == 'purchase', 1).otherwise(0))

df_features.show(10)

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+----+-----------+----------------+---------+-------------+---------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|hour|day_of_week|day_of_week_name|week_hour|added_to_cart|purchased|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+----+-----------+----------------+---------+-------------+---------+
|2019-09-30 17:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|  17|          2|          Monday|     2_17|            0|        0|
|2019-09-30 17:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|  17|          2|          Monday|     2_17|            0|      

In [10]:
# Check event types distribution
print('Event type distribution:')
df_features.groupBy('event_type').count().show()

print('Cart additions ratio:')
df_features.agg(avg('added_to_cart')).show()

print('Purchase ratio:')
df_features.agg(avg('purchased')).show()

Event type distribution:
+----------+---------+
|event_type|    count|
+----------+---------+
|  purchase|  1659788|
|      view|104335509|
|      cart|  3955446|
+----------+---------+

Cart additions ratio:
+--------------------+
|  avg(added_to_cart)|
+--------------------+
|0.035974709147713536|
+--------------------+

Purchase ratio:
+--------------------+
|      avg(purchased)|
+--------------------+
|0.015095741554015692|
+--------------------+



## 3. Unsupervised Learning - Customer Segmentation (Clustering)

In [11]:
# Aggregate user behavior for clustering
user_stats = df_features.groupBy('user_id').agg(
    count('*').alias('total_events'),
    avg('price').alias('avg_price'),
    sum('added_to_cart').alias('total_carts'),
    sum('purchased').alias('total_purchases'),
    approx_percentile('price', 0.5).alias('median_price')
)

# Calculate cart and purchase rates
user_stats = user_stats.withColumn('cart_rate', col('total_carts') / col('total_events')) \
    .withColumn('purchase_rate', col('total_purchases') / col('total_events')) \
    .fillna(0)

user_stats.show(10)

+---------+------------+------------------+-----------+---------------+------------+-------------------+-------------+
|  user_id|total_events|         avg_price|total_carts|total_purchases|median_price|          cart_rate|purchase_rate|
+---------+------------+------------------+-----------+---------------+------------+-------------------+-------------+
| 44378150|           1|            128.45|          0|              0|      128.45|                0.0|          0.0|
|116566414|          25|          152.0016|          0|              0|      163.19|                0.0|          0.0|
|122384079|           9| 98.22555555555556|          0|              0|      102.94|                0.0|          0.0|
|125917727|          24|           99.3925|          4|              0|       85.75|0.16666666666666666|          0.0|
|128130875|           1|            110.66|          0|              0|      110.66|                0.0|          0.0|
|146333366|           5|            246.08|     

In [12]:
# Prepare features for KMeans clustering
from pyspark.ml.feature import StandardScaler, VectorAssembler

feature_cols = ['total_events', 'avg_price', 'total_carts', 'total_purchases', 'cart_rate', 'purchase_rate']

# Assemble features into a vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
user_features = assembler.transform(user_stats)

# Standardize features
scaler = StandardScaler(inputCol='features', outputCol='scaled_features', withMean=True, withStd=True)
scaler_model = scaler.fit(user_features)
user_features_scaled = scaler_model.transform(user_features)

user_features_scaled.show(5)

+---------+------------+-----------------+-----------+---------------+------------+-------------------+-------------+--------------------+--------------------+
|  user_id|total_events|        avg_price|total_carts|total_purchases|median_price|          cart_rate|purchase_rate|            features|     scaled_features|
+---------+------------+-----------------+-----------+---------------+------------+-------------------+-------------+--------------------+--------------------+
| 44378150|           1|           128.45|          0|              0|      128.45|                0.0|          0.0|(6,[0,1],[1.0,128...|[-0.3650159113910...|
|116566414|          25|         152.0016|          0|              0|      163.19|                0.0|          0.0|(6,[0,1],[25.0,15...|[0.08011505640565...|
|122384079|           9|98.22555555555556|          0|              0|      102.94|                0.0|          0.0|(6,[0,1],[9.0,98....|[-0.2166389221254...|
|125917727|          24|          99.392

In [13]:
# Apply KMeans clustering
kmeans = KMeans(k=3, seed=42, featuresCol='scaled_features')
kmeans_model = kmeans.fit(user_features_scaled)
predictions = kmeans_model.transform(user_features_scaled)

# Show results
print('Cluster distribution:')
predictions.groupBy('prediction').count().show()

predictions.select('total_events', 'avg_price', 'cart_rate', 'purchase_rate', 'prediction').show(15)

Cluster distribution:
+----------+-------+
|prediction|  count|
+----------+-------+
|         1|  74991|
|         2| 425089|
|         0|4816569|
+----------+-------+

+------------+------------------+--------------------+--------------------+----------+
|total_events|         avg_price|           cart_rate|       purchase_rate|prediction|
+------------+------------------+--------------------+--------------------+----------+
|          50|175.18259999999998|                0.24|                 0.0|         2|
|          72|162.44180555555556|0.013888888888888888|0.013888888888888888|         0|
|           8|1130.8700000000001|                 0.0|                 0.0|         0|
|           1|            180.16|                 0.0|                 0.0|         0|
|           2|             82.55|                 0.0|                 0.0|         0|
|          10|            47.288|                 0.1|                 0.1|         2|
|          51| 203.6292156862745|0.058823529411

In [14]:
# Analyze clusters
print('Cluster 0 characteristics:')
predictions.filter(col('prediction') == 0).select('total_events', 'cart_rate', 'purchase_rate').describe().show()

print('Cluster 1 characteristics:')
predictions.filter(col('prediction') == 1).select('total_events', 'cart_rate', 'purchase_rate').describe().show()

print('Cluster 2 characteristics:')
predictions.filter(col('prediction') == 2).select('total_events', 'cart_rate', 'purchase_rate').describe().show()

Cluster 0 characteristics:
+-------+------------------+--------------------+--------------------+
|summary|      total_events|           cart_rate|       purchase_rate|
+-------+------------------+--------------------+--------------------+
|  count|           4816569|             4816569|             4816569|
|   mean|16.821025298298437| 0.00780521682536427|0.002169243769213...|
| stddev| 32.26129912418966|0.026568859980214285|0.010562038659069961|
|    min|                 1|                 0.0|                 0.0|
|    max|               402|                 0.2|               0.125|
+-------+------------------+--------------------+--------------------+

Cluster 1 characteristics:
+-------+------------------+-------------------+-------------------+
|summary|      total_events|          cart_rate|      purchase_rate|
+-------+------------------+-------------------+-------------------+
|  count|             74991|              74991|              74991|
|   mean|279.01229480871035|0.

## 4. Supervised Learning - Predict Probability of Adding to Cart (Temporal Features)

In [15]:
# Prepare data for cart prediction model
# Target: added_to_cart, Features: day_of_week, hour, price, category

# Filter out missing values
df_model = df_features.filter((col('day_of_week').isNotNull()) & 
                             (col('hour').isNotNull()) & 
                             (col('price').isNotNull())) \
    .select('day_of_week', 'hour', 'price', 'category_id', 'brand', 'added_to_cart')

print('Model dataset rows:', df_model.count())
df_model.show(10)

Model dataset rows: 109950743
+-----------+----+-------+-------------------+--------+-------------+
|day_of_week|hour|  price|        category_id|   brand|added_to_cart|
+-----------+----+-------+-------------------+--------+-------------+
|          2|  17|  35.79|2103807459595387724|shiseido|            0|
|          2|  17|   33.2|2053013552326770905|    aqua|            0|
|          2|  17|  543.1|2053013559792632471|    NULL|            0|
|          2|  17| 251.74|2053013558920217191|  lenovo|            0|
|          2|  17|1081.98|2053013555631882655|   apple|            0|
|          2|  17| 908.62|2053013561092866779|  pulser|            0|
|          2|  17| 380.96|2053013553853497655|   creed|            0|
|          2|  17|  41.16|2053013558031024687|luminarc|            0|
|          2|  17| 102.71|2053013565480109009|   baden|            0|
|          2|  17| 566.01|2053013555631882655|  huawei|            0|
+-----------+----+-------+-------------------+--------+-----

In [16]:
# Encode categorical variables
cat_indexers = [StringIndexer(inputCol=col, outputCol=col+'_indexed').fit(df_model) 
                for col in ['category_id', 'brand']]

# Apply indexing
df_indexed = df_model
for indexer in cat_indexers:
    df_indexed = indexer.transform(df_indexed)

df_indexed.show(10)

Py4JJavaError: An error occurred while calling o549.showString.
: org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`StringIndexerModel$$Lambda$7221/0x000001c63660a3e8`: (string) => double) failed due to: org.apache.spark.SparkException: StringIndexer encountered NULL value. To handle or skip NULLS, try setting StringIndexer.handleInvalid.. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:197)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage3.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFact

In [ ]:
# Create feature vector for cart prediction
feature_cols_cart = ['day_of_week', 'hour', 'price', 'category_id_indexed', 'brand_indexed']
assembler_cart = VectorAssembler(inputCols=feature_cols_cart, outputCol='features')
df_cart_features = assembler_cart.transform(df_indexed).select('features', 'added_to_cart')

# Standardize features
scaler_cart = StandardScaler(inputCol='features', outputCol='scaled_features', withMean=True, withStd=True)
scaler_cart_model = scaler_cart.fit(df_cart_features)
df_cart_scaled = scaler_cart_model.transform(df_cart_features)

print('Scaled features created')
df_cart_scaled.show(5)

In [ ]:
# Split data for training and testing
train_cart, test_cart = df_cart_scaled.randomSplit([0.8, 0.2], seed=42)

print('Training set:', train_cart.count(), 'rows')
print('Testing set:', test_cart.count(), 'rows')

In [ ]:
# Train Logistic Regression for cart prediction
lr = LogisticRegression(featuresCol='scaled_features', labelCol='added_to_cart', maxIter=100)
lr_model = lr.fit(train_cart)

# Make predictions
lr_predictions = lr_model.transform(test_cart)

# Evaluate
evaluator = BinaryClassificationEvaluator(labelCol='added_to_cart')
auc_lr = evaluator.evaluate(lr_predictions)

print('Logistic Regression AUC:', auc_lr)
lr_predictions.select('added_to_cart', 'probability', 'prediction').show(10)

In [ ]:
# Train Random Forest for cart prediction
rf_cart = RandomForestClassifier(numTrees=10, maxDepth=5, seed=42, 
                                featuresCol='scaled_features', labelCol='added_to_cart')
rf_cart_model = rf_cart.fit(train_cart)

# Make predictions
rf_cart_predictions = rf_cart_model.transform(test_cart)

# Evaluate
auc_rf_cart = evaluator.evaluate(rf_cart_predictions)

print('Random Forest AUC (Cart Prediction):', auc_rf_cart)
rf_cart_predictions.select('added_to_cart', 'probability', 'prediction').show(10)

In [ ]:
# Feature importance
print('Random Forest Feature Importance:')
importance_df = pd.DataFrame({
    'feature': feature_cols_cart,
    'importance': rf_cart_model.featureImportances.toArray()
})
print(importance_df.sort_values('importance', ascending=False))

## 5. Supervised Learning - Predict Sales by Brand (Regression)

In [ ]:
# Aggregate sales by brand
brand_sales = df_features.filter(col('event_type') == 'purchase') \
    .groupBy('brand').agg(
        count('*').alias('total_purchases'),
        sum('price').alias('total_revenue'),
        avg('price').alias('avg_price')
    ) \
    .filter(col('brand').isNotNull())

brand_sales = brand_sales.withColumn('revenue_per_purchase', col('total_revenue') / col('total_purchases'))

print('Top brands by revenue:')
brand_sales.sort(desc('total_revenue')).show(10)

In [ ]:
# Create features for sales prediction by brand
# Including: average price, cart rate per brand, number of SKUs, etc.

brand_features = df_features.groupBy('brand').agg(
    count('*').alias('total_events'),
    sum('added_to_cart').alias('total_carts'),
    sum('purchased').alias('total_purchases'),
    avg('price').alias('avg_price'),
    countDistinct('product_id').alias('num_products')
) \
    .filter(col('brand').isNotNull())

brand_features = brand_features.withColumn('cart_rate', col('total_carts') / col('total_events')) \
    .withColumn('purchase_rate', col('total_purchases') / col('total_events')) \
    .fillna(0)

print('Brand features:')
brand_features.show(10)

In [ ]:
# Merge sales with features
brand_model_data = brand_features.join(brand_sales.select('brand', 'total_revenue'), 'brand')

print('Brand model data:')
brand_model_data.show(10)

In [ ]:
# Prepare features for regression
feature_cols_sales = ['avg_price', 'cart_rate', 'purchase_rate', 'num_products']
assembler_sales = VectorAssembler(inputCols=feature_cols_sales, outputCol='features')
brand_features_vec = assembler_sales.transform(brand_model_data).select('features', 'total_revenue')

# Standardize
scaler_sales = StandardScaler(inputCol='features', outputCol='scaled_features', withMean=True, withStd=True)
scaler_sales_model = scaler_sales.fit(brand_features_vec)
brand_features_scaled = scaler_sales_model.transform(brand_features_vec)

# Split data
train_sales, test_sales = brand_features_scaled.randomSplit([0.8, 0.2], seed=42)

print('Training set:', train_sales.count())
print('Testing set:', test_sales.count())

In [ ]:
# Train Linear Regression
lr_sales = LinearRegression(featuresCol='scaled_features', labelCol='total_revenue', maxIter=100)
lr_sales_model = lr_sales.fit(train_sales)

# Make predictions
lr_sales_predictions = lr_sales_model.transform(test_sales)

# Evaluate
evaluator_reg = RegressionEvaluator(labelCol='total_revenue', predictionCol='prediction', metricName='r2')
r2_lr = evaluator_reg.evaluate(lr_sales_predictions)

print('Linear Regression R²:', r2_lr)
lr_sales_predictions.select('total_revenue', 'prediction').show(10)

In [ ]:
# Train Random Forest Regressor
rf_sales = RandomForestRegressor(numTrees=10, maxDepth=5, seed=42, 
                                featuresCol='scaled_features', labelCol='total_revenue')
rf_sales_model = rf_sales.fit(train_sales)

# Make predictions
rf_sales_predictions = rf_sales_model.transform(test_sales)

# Evaluate
r2_rf = evaluator_reg.evaluate(rf_sales_predictions)

print('Random Forest R² (Sales Prediction):', r2_rf)
rf_sales_predictions.select('total_revenue', 'prediction').show(10)

In [ ]:
# Feature importance for sales prediction
print('Random Forest Feature Importance (Sales):')
importance_sales_df = pd.DataFrame({
    'feature': feature_cols_sales,
    'importance': rf_sales_model.featureImportances.toArray()
})
print(importance_sales_df.sort_values('importance', ascending=False))

## 6. Model Comparison and Summary

In [ ]:
# Summary of models
print('=== CART PREDICTION MODELS ===')
print(f'Logistic Regression AUC: {auc_lr:.4f}')
print(f'Random Forest AUC: {auc_rf_cart:.4f}')

print('=== SALES PREDICTION MODELS ===')
print(f'Linear Regression R²: {r2_lr:.4f}')
print(f'Random Forest R²: {r2_rf:.4f}')

print('=== CUSTOMER SEGMENTATION ===')
print(f'Number of clusters: 3')
print('Segments identified based on purchasing behavior and engagement')

## 7. Recommendations and Next Steps

### Key Insights:
1. **Customer Segmentation** identifies three distinct customer groups with different behaviors
2. **Temporal Patterns** show how day and hour affect cart addition probability
3. **Brand Performance** can be predicted based on engagement metrics

### Next Steps:
- Hyperparameter tuning using cross-validation
- Deploy best models for real-time predictions
- Monitor model drift over time
- A/B test personalized recommendations based on cluster assignments